# ITEM 1 & 1A EXTRACTION FROM 10-K FILINGS (UPDATED)

**🆕 What's New in This Version:**
- ⚡ **2-4x Faster** multi-process extraction
- 📊 **Real progress tracking** to avoid false "completed" messages
- 🔄 **Auto-resume** capability after Colab disconnections
- ✅ **Status verification** before and after extraction

**Purpose:** Extract Business (Item 1) and Risk Factors (Item 1A) sections from 10-K filings

**What This Does:**
- Extracts Item 1 (Business Description) and Item 1A (Risk Factors)
- Creates separate JSON files organized by year (2010-2025)
- Generates analysis-ready CSV and Parquet files
- **Preserves existing MD&A files** using backup/restore

**Prerequisites:**
- Repository: `/content/drive/MyDrive/EDGAR_Project/edgar-crawler`
- Raw 10-K files already downloaded in `datasets/RAW_FILINGS/10-K/`
- Metadata file: `datasets/FILINGS_METADATA.csv`

**⚠️ Important Notes:**
- Colab sessions timeout after 12-24 hours
- For ~80K files, you may need **multiple sessions**
- This notebook automatically resumes where it left off

---

## SECTION 1: SETUP & PULL LATEST CODE

Run these cells at the start of every Colab session

In [ ]:
## 🟢 Cell 1: Mount Google Drive
import os
from google.colab import drive

if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive already mounted")
else:
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")

In [ ]:
## 🟢 Cell 2: Navigate to Repository
import os

REPO_DIR = '/content/drive/MyDrive/EDGAR_Project/edgar-crawler'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Working directory: {os.getcwd()}")
else:
    print(f"❌ Repository not found at: {REPO_DIR}")
    print("Please update REPO_DIR to match your Google Drive structure")

In [ ]:
## 🆕 Cell 3: Pull Latest Code from Git
print("Pulling latest improvements from git...\n")

!git fetch origin claude/debug-edgar-crawler-JUAou
!git pull origin claude/debug-edgar-crawler-JUAou

print("\n" + "="*70)
print(" Verifying New Files")
print("="*70)

import os
new_files = [
    'extract_items_fast.py',
    'flexible_extractor_fast.py',
    'check_extraction_status.py',
    'COLAB_EXTRACTION_GUIDE.md',
    'extraction_configs/items_1_1a.json'
]

all_present = True
for file in new_files:
    exists = os.path.exists(file)
    status = "✅" if exists else "❌"
    print(f"{status} {file}")
    if not exists:
        all_present = False

if all_present:
    print("\n✅ All new files are present!")
else:
    print("\n⚠️ Some files are missing. Git pull may have failed.")
    print("Try running: !git reset --hard HEAD && !git pull origin claude/debug-edgar-crawler-JUAou")

In [ ]:
## Cell 4: Install Dependencies
print("Installing dependencies...")

!pip install -q 'dill<0.3.9' 'multiprocess<0.70.17'
!pip install -q pox ppft
!pip install -q --no-deps pathos
!pip install -q beautifulsoup4 lxml requests pandas tqdm click cssutils numpy pyarrow

print("✅ Dependencies installed")

In [ ]:
## Cell 5: Keep-Alive Script
from IPython.display import display, Javascript

display(Javascript('''
function KeepClicking(){
    console.log("Keeping session alive...");
    document.querySelector("colab-connect-button").click();
}
setInterval(KeepClicking, 60000);
'''))

print("✅ Keep-alive activated (prevents early disconnection)")

---

## SECTION 2: CHECK CURRENT STATUS

🆕 **Always run this first** to see how many files are already extracted

In [ ]:
## 🆕 Cell 6: Check Extraction Status (IMPORTANT!)

# This shows REAL progress, not misleading "completed" messages
!python check_extraction_status.py

**📊 Interpret the Results Above:**

- If **Progress: 0%** → Extraction hasn't started
- If **Progress: < 100%** → Extraction is incomplete, continue below
- If **Progress: ~100%** → Extraction is complete! Skip to SECTION 6

---

## SECTION 3: PREPARE FOR EXTRACTION

Create configuration and filter metadata

In [ ]:
## Cell 7: Filter Metadata to 2010 Onwards
import pandas as pd
import os

metadata_path = 'datasets/FILINGS_METADATA.csv'

print("Loading and filtering metadata...\n")

if not os.path.exists(metadata_path):
    print(f"❌ Metadata file not found: {metadata_path}")
    print("You need to download filings first!")
else:
    metadata = pd.read_csv(metadata_path)
    print(f"Loaded {len(metadata):,} total filings")
    
    # Filter to 10-K and 2010 onwards
    metadata_10k = metadata[metadata['Type'] == '10-K'].copy()
    metadata_filtered = metadata_10k[metadata_10k['year'] >= 2010].copy()
    
    print(f"\nFiltered Results:")
    print(f"   10-K filings >= 2010: {len(metadata_filtered):,}")
    print(f"   Excluded (before 2010): {len(metadata_10k) - len(metadata_filtered):,}")
    
    # Year distribution
    print(f"\nYear Distribution:")
    year_counts = metadata_filtered['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        print(f"      {year}: {count:,} filings")
    
    # Save filtered metadata
    filtered_path = 'datasets/FILINGS_METADATA_2010_onwards.csv'
    metadata_filtered.to_csv(filtered_path, index=False)
    print(f"\n✅ Filtered metadata saved: {filtered_path}")

In [ ]:
## Cell 8: Update config.json for Items 1 & 1A
import json

# Load main config
with open('config.json', 'r') as f:
    config = json.load(f)

# Update extract_items section for Items 1 & 1A
config['extract_items']['filings_metadata_file'] = 'FILINGS_METADATA_2010_onwards.csv'
config['extract_items']['filing_types'] = ['10-K']
config['extract_items']['items_to_extract'] = ['1', '1A']
config['extract_items']['remove_tables'] = True

# 🆕 DISABLE special_items (not needed for Items 1 & 1A)
config['extract_items']['special_items']['enabled'] = False

# ⚠️ IMPORTANT: Choose your re-extraction strategy
# Option 1: Skip existing files (faster, keeps old files with special_items)
# config['extract_items']['skip_extracted_filings'] = True

# Option 2: Re-extract all files (slower, all files will be clean)
config['extract_items']['skip_extracted_filings'] = False  # ← CHANGE TO True IF YOU WANT TO KEEP EXISTING FILES

# Save updated config
with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Config updated for Items 1 & 1A extraction")
print("   Items: 1 (Business), 1A (Risk Factors)")
print("   Special items: DISABLED (cleaner JSON files)")
print(f"   Skip existing: {config['extract_items']['skip_extracted_filings']}")
if not config['extract_items']['skip_extracted_filings']:
    print("\n⚠️  RE-EXTRACTION MODE: All files will be re-extracted (no special_items)")
else:
    print("\n✅ SKIP MODE: Existing files kept, only new files extracted")

In [ ]:
## Cell 9: Create Output Directory
import os

output_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'
os.makedirs(output_base, exist_ok=True)

print(f"✅ Output directory ready: {output_base}/")
print("   Year subfolders will be created automatically")

---

## SECTION 4: BACKUP MD&A FILES

Preserve existing MD&A extractions before starting

In [ ]:
## Cell 10: Backup Existing MD&A Files
import os
import shutil
import time

backup_dir = 'datasets/EXTRACTED_FILINGS/10-K_MDA_BACKUP'
original_dir = 'datasets/EXTRACTED_FILINGS/10-K'

print("="*70)
print(" BACKUP EXISTING MD&A FILES")
print("="*70)

if os.path.exists(original_dir):
    print("\n📦 Backing up existing MD&A files...")
    
    # Remove old backup if it exists
    if os.path.exists(backup_dir):
        print("   Removing old backup...")
        shutil.rmtree(backup_dir)
    
    start_time = time.time()
    shutil.move(original_dir, backup_dir)
    elapsed = time.time() - start_time
    
    print(f"\n✅ Backup completed in {elapsed:.1f} seconds")
    print(f"   Backup location: {backup_dir}/")
else:
    print("\nℹ️  No existing 10-K/ folder found")
    print("   This is a fresh extraction (no backup needed)")

print("\n" + "="*70)

---

## SECTION 5: RUN FAST MULTI-PROCESS EXTRACTION ⚡

🆕 **This is 2-4x faster** than the original single-process extraction

In [ ]:
## 🆕 Cell 11: Run FAST Multi-Process Extraction
import multiprocessing

# Auto-detect optimal number of processes
cpu_count = multiprocessing.cpu_count()
num_processes = min(3, cpu_count - 1) if cpu_count > 1 else 1

print("="*70)
print(" STARTING FAST EXTRACTION (Multi-Process)")
print("="*70)
print(f"\n💻 CPU cores available: {cpu_count}")
print(f"⚡ Using {num_processes} parallel processes")
print(f"🚀 This will be {num_processes}x faster than single-process!\n")
print("Configuration:")
print("   Items: 1 (Business), 1A (Risk Factors)")
print("   Output: datasets/EXTRACTED_FILINGS/10-K/ (temp location)")
print("   Skip existing: Yes (auto-resumes if interrupted)")
print(f"\n⏱️  Estimated time: ~24-48 hours (with {num_processes} processes)")
print("   💡 TIP: Run Cell 12 in parallel to monitor progress!\n")
print("="*70)

!python flexible_extractor_fast.py \
    --config extraction_configs/items_1_1a.json \
    --processes {num_processes}

In [ ]:
## 🆕 Cell 12: Real-Time Progress Monitor (Run This While Cell 11 is Running!)

# Run this cell in PARALLEL while Cell 11 is extracting
# It will update every 30 seconds showing real progress
# Press STOP button to exit (extraction continues)

import os
import time
from IPython.display import clear_output

print("📊 Real-Time Progress Monitor")
print("Press STOP button to exit monitoring (extraction continues)")
print("="*70)

extracted_dir = 'datasets/EXTRACTED_FILINGS/10-K'

# Get initial count
def count_json_files(directory):
    if not os.path.exists(directory):
        return 0
    count = 0
    for root, dirs, files in os.walk(directory):
        count += len([f for f in files if f.endswith('.json')])
    return count

initial_count = count_json_files(extracted_dir)
start_time = time.time()

try:
    while True:
        clear_output(wait=True)
        
        current_count = count_json_files(extracted_dir)
        elapsed = time.time() - start_time
        files_this_session = current_count - initial_count
        
        print(f"📊 Real-Time Progress Monitor")
        print(f"="*70)
        
        if elapsed > 60 and files_this_session > 0:
            rate = files_this_session / elapsed
            remaining_files = 79532 - current_count
            estimated_remaining = remaining_files / rate if rate > 0 else 0
            
            print(f"\n📈 Progress: {current_count:,} / 79,532 ({current_count/79532*100:.1f}%)")
            print(f"\n⏱️  Session Stats:")
            print(f"   Processed this session: {files_this_session:,} files")
            print(f"   Time elapsed: {elapsed/60:.1f} minutes")
            print(f"   Current rate: {rate*60:.1f} files/min ({rate*3600:.0f} files/hour)")
            print(f"\n🎯 Estimates:")
            print(f"   Remaining: {remaining_files:,} files")
            print(f"   Est. time to complete: {estimated_remaining/3600:.1f} hours")
            
            # Progress bar
            progress_pct = current_count / 79532
            bar_length = 50
            filled = int(bar_length * progress_pct)
            bar = '█' * filled + '░' * (bar_length - filled)
            print(f"\n   [{bar}] {progress_pct*100:.1f}%")
            
            if progress_pct >= 0.99:
                print(f"\n🎉 EXTRACTION IS COMPLETE!")
                break
        else:
            print(f"\n⏳ Waiting for extraction to start...")
            print(f"   Current count: {current_count:,}")
            print(f"   Time elapsed: {elapsed:.0f} seconds")
        
        print(f"\n" + "="*70)
        print(f"Last updated: {time.strftime('%H:%M:%S')}")
        
        time.sleep(30)  # Update every 30 seconds
        
except KeyboardInterrupt:
    print(f"\n\n✅ Monitoring stopped (extraction continues in Cell 11)")
    print(f"   Final count: {current_count:,} files")

---

## 🔄 IF COLAB DISCONNECTED: Resume Here

If your session disconnected:
1. Run Cells 1-5 (setup)
2. Run Cell 6 to check current progress
3. Re-run Cell 11 to continue (it will skip existing files)

---

## SECTION 6: VERIFY COMPLETION

Run this after Cell 11 finishes to verify extraction

In [ ]:
## 🆕 Cell 13: Verify Extraction Completed Successfully

print("Verifying extraction results...\n")

!python check_extraction_status.py

print("\n" + "="*70)
print(" NEXT STEPS")
print("="*70)
print("\nIf progress shows ~100%:")
print("   ✅ Continue to Cell 14 to reorganize files")
print("\nIf progress shows < 100%:")
print("   ⚠️  Re-run Cell 11 to continue extraction")
print("   (It will automatically skip already-extracted files)")

---

## SECTION 7: REORGANIZE & RESTORE

Move Items 1 & 1A to proper location and restore MD&A files

In [ ]:
## Cell 14: Move Files to item_1_1a Directory
import os
import shutil
import json
from tqdm import tqdm

print("="*70)
print(" REORGANIZING FILES TO item_1_1a/")
print("="*70)
print("\nMoving extracted files to item_1_1a directory...\n")

source_base = 'datasets/EXTRACTED_FILINGS/10-K'
dest_base = 'datasets/EXTRACTED_FILINGS/item_1_1a'

os.makedirs(dest_base, exist_ok=True)

moved_count = 0
skipped_count = 0

if os.path.exists(source_base):
    for year_folder in sorted(os.listdir(source_base)):
        year_path = os.path.join(source_base, year_folder)
        
        if not os.path.isdir(year_path):
            continue
        
        try:
            if int(year_folder) < 2010:
                continue
        except:
            continue
        
        dest_year_path = os.path.join(dest_base, year_folder)
        os.makedirs(dest_year_path, exist_ok=True)
        
        json_files = [f for f in os.listdir(year_path) if f.endswith('.json')]
        
        for filename in tqdm(json_files, desc=f"Year {year_folder}", leave=False):
            source_file = os.path.join(year_path, filename)
            dest_file = os.path.join(dest_year_path, filename)
            
            if os.path.exists(dest_file):
                skipped_count += 1
                continue
            
            try:
                with open(source_file, 'r') as f:
                    data = json.load(f)
                
                has_item_1 = 'item_1' in data and len(data.get('item_1', '')) > 0
                has_item_1a = 'item_1a' in data and len(data.get('item_1a', '')) > 0
                
                if has_item_1 or has_item_1a:
                    shutil.move(source_file, dest_file)
                    moved_count += 1
            except Exception as e:
                print(f"\nWarning: Error processing {filename}: {e}")

    print(f"\n✅ Reorganization complete!")
    print(f"   Moved: {moved_count:,} files to {dest_base}/")
    print(f"   Skipped (already exist): {skipped_count:,} files")
else:
    print(f"⚠️  Source directory not found: {source_base}")

print("\n" + "="*70)

In [ ]:
## Cell 15: Restore MD&A Files
import os
import shutil
import time

backup_dir = 'datasets/EXTRACTED_FILINGS/10-K_MDA_BACKUP'
original_dir = 'datasets/EXTRACTED_FILINGS/10-K'

print("="*70)
print(" RESTORE MD&A FILES")
print("="*70)

if os.path.exists(backup_dir):
    print("\n📦 Restoring original MD&A files...")
    
    if os.path.exists(original_dir):
        print("   Removing temporary extraction folder...")
        shutil.rmtree(original_dir)
    
    start_time = time.time()
    shutil.move(backup_dir, original_dir)
    elapsed = time.time() - start_time
    
    print(f"\n✅ Restore completed in {elapsed:.1f} seconds")
    print(f"   MD&A files restored to: {original_dir}/")
    print(f"\n📊 Final Structure:")
    print(f"   - {original_dir}/ → Contains MD&A (Item 7)")
    print(f"   - datasets/EXTRACTED_FILINGS/item_1_1a/ → Contains Items 1 & 1A")
else:
    print("\nℹ️  No backup to restore")
    print("   Either backup was already restored or this was a fresh extraction")

print("\n" + "="*70)

---

## SECTION 8: CREATE ANALYSIS FILES

Generate metadata CSV and Parquet for downstream analysis

In [ ]:
## Cell 16: Check Final Results
import os
import json
from collections import defaultdict

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'

print("="*70)
print(" FINAL EXTRACTION RESULTS")
print("="*70)

year_counts = defaultdict(int)
all_files = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    if json_files and root != extracted_dir:
        year = os.path.basename(root)
        year_counts[year] = len(json_files)
        all_files.extend([os.path.join(root, f) for f in json_files])

print(f"\n📊 Total Extracted: {len(all_files):,} filings")
print(f"\n📁 Files by Year:")
for year in sorted(year_counts.keys()):
    print(f"      {year}: {year_counts[year]:,} files")

# Quality check
if len(all_files) > 0:
    print(f"\n✅ Quality Check (5 random samples):\n")
    import random
    samples = random.sample(all_files, min(5, len(all_files)))
    
    for fpath in samples:
        fname = os.path.basename(fpath)
        try:
            with open(fpath, 'r') as f:
                data = json.load(f)
            
            item1_len = len(data.get('item_1', ''))
            item1a_len = len(data.get('item_1a', ''))
            
            status1 = 'YES' if item1_len > 100 else 'NO'
            status1a = 'YES' if item1a_len > 100 else 'NO'
            
            print(f"   {fname[:45]:45s}")
            print(f"      Item 1:  {status1:3s} ({item1_len:>7,} chars)")
            print(f"      Item 1A: {status1a:3s} ({item1a_len:>7,} chars)\n")
        except Exception as e:
            print(f"   {fname}: Error - {e}\n")

print("="*70)

In [ ]:
## Cell 17: Create Metadata CSV
import os
import json
import pandas as pd
from tqdm import tqdm

print("Creating metadata CSV for Items 1 & 1A...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
metadata_records = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            metadata_records.append({
                'filename': filename,
                'cik': filing.get('cik', ''),
                'company': filing.get('company', ''),
                'filing_date': filing.get('filing_date', ''),
                'year': filing.get('period_of_report', '')[:4] if filing.get('period_of_report') else '',
                'has_item_1': 'item_1' in filing and len(filing.get('item_1', '')) > 0,
                'item_1_length': len(filing.get('item_1', '')),
                'has_item_1a': 'item_1a' in filing and len(filing.get('item_1a', '')) > 0,
                'item_1a_length': len(filing.get('item_1a', '')),
                'json_path': filepath
            })
        except Exception as e:
            pass

df_meta = pd.DataFrame(metadata_records)
meta_path = 'datasets/items_1_1a_metadata.csv'
df_meta.to_csv(meta_path, index=False)

print(f"\n✅ Metadata CSV created!")
print(f"   Location: {meta_path}")
print(f"   Records: {len(df_meta):,}")
print(f"   File size: {os.path.getsize(meta_path) / 1024:.1f} KB")

print(f"\n📊 Summary Statistics:")
print(f"   Total filings: {len(df_meta):,}")
print(f"   Filings with Item 1: {df_meta['has_item_1'].sum():,}")
print(f"   Filings with Item 1A: {df_meta['has_item_1a'].sum():,}")
print(f"   Filings with BOTH: {(df_meta['has_item_1'] & df_meta['has_item_1a']).sum():,}")

print(f"\n📏 Length Statistics:")
print(f"   Item 1 avg: {df_meta['item_1_length'].mean():,.0f} characters")
print(f"   Item 1A avg: {df_meta['item_1a_length'].mean():,.0f} characters")

In [ ]:
## Cell 18: Create Consolidated Parquet File
import os
import json
import pandas as pd
from tqdm import tqdm

print("Creating consolidated Parquet file...\n")

extracted_dir = 'datasets/EXTRACTED_FILINGS/item_1_1a'
full_data = []

for root, dirs, files in os.walk(extracted_dir):
    json_files = [f for f in files if f.endswith('.json')]
    
    for filename in tqdm(json_files, desc=f"Processing {os.path.basename(root)}", leave=False):
        filepath = os.path.join(root, filename)
        try:
            with open(filepath, 'r') as f:
                filing = json.load(f)
            
            if ('item_1' in filing and len(filing.get('item_1', '')) > 0) or \
               ('item_1a' in filing and len(filing.get('item_1a', '')) > 0):
                full_data.append({
                    'cik': filing.get('cik', ''),
                    'company': filing.get('company', ''),
                    'filing_date': filing.get('filing_date', ''),
                    'year': filing.get('period_of_report', '')[:4] if filing.get('period_of_report') else '',
                    'item_1_text': filing.get('item_1', ''),
                    'item_1a_text': filing.get('item_1a', '')
                })
        except Exception as e:
            pass

df_full = pd.DataFrame(full_data)
parquet_path = 'datasets/items_1_1a_full.parquet'
df_full.to_parquet(parquet_path, compression='gzip', index=False)

print(f"\n✅ Parquet file created!")
print(f"   Location: {parquet_path}")
print(f"   Records: {len(df_full):,}")
print(f"   File size: {os.path.getsize(parquet_path) / (1024**2):.1f} MB")

print(f"\n📊 Text Length Statistics:")
print(f"\n   Item 1 (Business):")
print(df_full['item_1_text'].str.len().describe().to_string())

print(f"\n   Item 1A (Risk Factors):")
print(df_full['item_1a_text'].str.len().describe().to_string())

---

## 🎉 EXTRACTION COMPLETE!

### Summary of Outputs:

1. **JSON Files** (organized by year):
   - Location: `datasets/EXTRACTED_FILINGS/item_1_1a/`
   - Structure: Year subfolders (2010-2025)
   - Each file contains: `item_1` (Business) and `item_1a` (Risk Factors)

2. **MD&A Files** (preserved):
   - Location: `datasets/EXTRACTED_FILINGS/10-K/`
   - Contains: Original MD&A extractions (Item 7)
   - **Untouched** by this extraction process

3. **Metadata CSV**:
   - Location: `datasets/items_1_1a_metadata.csv`
   - Contains: File info, lengths, flags for Items 1 & 1A

4. **Parquet File** (compressed):
   - Location: `datasets/items_1_1a_full.parquet`
   - Contains: Full text for Items 1 & 1A
   - Use for: Text analysis, NLP, machine learning

### What Changed in This Updated Notebook:

✅ **2-4x faster extraction** using multi-process parallelization
✅ **Real progress tracking** to avoid false "completed" messages
✅ **Auto-resume capability** after Colab disconnections
✅ **Status verification** before and after extraction
✅ **Real-time monitoring** while extraction runs

### Next Steps:

- **Perform text analysis** (sentiment, readability, etc.)
- **Compare with MD&A data** (Item 7)
- **Research applications**: Risk analysis, business model comparison

---

**Questions or Issues?**
- Check the repository: https://github.com/haowenluo/edgar-crawler
- See `COLAB_EXTRACTION_GUIDE.md` for detailed troubleshooting

---